# 🔲 Exercícios — Autômatos Celulares

**Disciplina:** Inteligência Artificial | **Nível:** Intermediário

> Explore a emergência de comportamento complexo a partir de regras locais simples.


## 1. Autômato Celular 1D — Regra de Wolfram

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def aplicar_regra_1d(estado, numero_regra):
    """Aplica a regra de Wolfram (0-255) a um estado 1D."""
    regra_bits = format(numero_regra, '08b')  # 8 bits da regra
    novo_estado = np.zeros_like(estado)
    n = len(estado)
    for i in range(n):
        esq = estado[(i-1)%n]
        cen = estado[i]
        dir_ = estado[(i+1)%n]
        padrao = esq*4 + cen*2 + dir_*1
        novo_estado[i] = int(regra_bits[7 - padrao])
    return novo_estado

def simular_1d(numero_regra, n_celulas=101, n_passos=50):
    estado = np.zeros(n_celulas, dtype=int)
    estado[n_celulas//2] = 1  # célula central ativa
    historia = [estado.copy()]
    for _ in range(n_passos):
        estado = aplicar_regra_1d(estado, numero_regra)
        historia.append(estado.copy())
    return np.array(historia)

# Regras famosas: 30 (caótica), 90 (fractal), 110 (universal), 184 (tráfego)
fig, axes = plt.subplots(2,2,figsize=(14,10))
for ax, regra in zip(axes.flat, [30, 90, 110, 184]):
    historia = simular_1d(regra, n_celulas=101, n_passos=50)
    ax.imshow(historia, cmap='binary', interpolation='nearest')
    ax.set_title(f'Regra {regra}', fontsize=13)
    ax.set_xlabel('Célula'); ax.set_ylabel('Passo')
plt.suptitle('Autômatos Celulares 1D — Regras de Wolfram', fontsize=14)
plt.tight_layout(); plt.show()


### 📝 Exercício 1

Explore **3 outras regras** de sua escolha (1-255). Classifique cada uma em: **estável, periódica, caótica ou complexa**.

Wolfram propõe 4 classes de comportamento. Você consegue identificar uma regra para cada classe?

In [ ]:
minhas_regras = [18, 54, 126]  # ✏️ Troque pelos números que quiser
fig, axes = plt.subplots(1, len(minhas_regras), figsize=(14,5))
for ax, regra in zip(axes, minhas_regras):
    h = simular_1d(regra)
    ax.imshow(h, cmap='binary', interpolation='nearest')
    ax.set_title(f'Regra {regra}')
plt.tight_layout(); plt.show()
# Classifique abaixo:
classificacoes = {r: "?" for r in minhas_regras}
print("Minhas classificações:", classificacoes)


## 2. Jogo da Vida de Conway (2D)

In [ ]:
def contar_vizinhos(grade):
    """Conta vizinhos vivos (8-conectividade) com convolução."""
    from numpy import roll
    vizinhos = sum(
        roll(roll(grade, i, 0), j, 1)
        for i in (-1,0,1) for j in (-1,0,1) if not (i==0 and j==0)
    )
    return vizinhos

def passo_jogo_vida(grade):
    """Aplica as regras do Jogo da Vida de Conway."""
    viz = contar_vizinhos(grade)
    # Sobrevivência: vivo com 2 ou 3 vizinhos
    sobrevive = grade & ((viz==2) | (viz==3))
    # Nascimento: morto com exatamente 3 vizinhos
    nasce = ~grade & (viz==3)
    return (sobrevive | nasce).astype(int)

# Configuração inicial: planar aleatório
np.random.seed(42)
grade = (np.random.rand(60,60) < 0.3).astype(int)

n_passos = 30
populacao_hist = [grade.sum()]
grades = [grade.copy()]
for _ in range(n_passos-1):
    grade = passo_jogo_vida(grade)
    grades.append(grade.copy())
    populacao_hist.append(grade.sum())

# Exibir passos selecionados
passos_exibir = [0, 5, 15, 29]
fig, axes = plt.subplots(1,4,figsize=(16,4))
for ax, p in zip(axes, passos_exibir):
    ax.imshow(grades[p], cmap='binary', interpolation='nearest')
    ax.set_title(f'Passo {p}  ({grades[p].sum()} vivas)'); ax.axis('off')
plt.suptitle('Jogo da Vida de Conway'); plt.tight_layout(); plt.show()

plt.figure(figsize=(8,3))
plt.plot(populacao_hist,'b-'); plt.title('População ao longo do tempo')
plt.xlabel('Passo'); plt.ylabel('Células vivas'); plt.grid(True); plt.show()


### 📝 Exercício 2

Insira padrões clássicos no Jogo da Vida:
1. **Blinker** (oscilador de período 2): `[[0,1,0],[0,1,0],[0,1,0]]`
2. **Glider** (se move diagonalmente)
3. **Block** (estático)

Observe por quantos passos cada um mantém seu padrão.

In [ ]:
def inserir_padrao(grade, padrao, linha, col):
    """Insere um padrão em uma posição específica."""
    g = grade.copy()
    for i, linha_p in enumerate(padrao):
        for j, val in enumerate(linha_p):
            g[linha+i, col+j] = val
    return g

# Padrões clássicos
blinker = [[0,1,0],[0,1,0],[0,1,0]]
glider  = [[0,1,0],[0,0,1],[1,1,1]]
block   = [[1,1],[1,1]]

# Grade vazia
grade_vazia = np.zeros((30,60), dtype=int)
grade_padroes = inserir_padrao(grade_vazia, blinker, 5, 5)
grade_padroes = inserir_padrao(grade_padroes, glider, 10, 20)
grade_padroes = inserir_padrao(grade_padroes, block, 5, 45)

fig, axes = plt.subplots(1, 4, figsize=(16,4))
g = grade_padroes.copy()
for ax, passo in zip(axes, [0,2,5,10]):
    while g.sum() > 0 and passo > 0:
        # simular até o passo desejado
        break
    ax.imshow(g, cmap='binary'); ax.set_title(f'Passo 0'); ax.axis('off')

# ✏️ Corrija a simulação: simule passo a passo e exiba os passos 0, 2, 5, 10
grades_p = [grade_padroes.copy()]
for _ in range(10):
    grades_p.append(passo_jogo_vida(grades_p[-1]))

fig, axes = plt.subplots(1,4,figsize=(16,4))
for ax, p in zip(axes, [0,2,5,10]):
    ax.imshow(grades_p[p],cmap='binary'); ax.set_title(f'Passo {p}'); ax.axis('off')
plt.suptitle('Padrões: Blinker, Glider, Block'); plt.tight_layout(); plt.show()


## 3. Autômato Celular para Difusão

Simule a difusão de uma substância em uma grade 2D.

In [ ]:
def difusao_2d(concentracao, D=0.25):
    """Passo de difusão por diferenças finitas."""
    laplaciano = (
        np.roll(concentracao,1,0) + np.roll(concentracao,-1,0) +
        np.roll(concentracao,1,1) + np.roll(concentracao,-1,1) - 
        4*concentracao
    )
    return concentracao + D*laplaciano

# Condição inicial: ponto de concentração máxima no centro
N = 50
C = np.zeros((N,N))
C[N//2,N//2] = 1.0

passos_dif = [0, 10, 30, 80]
Cs = [C.copy()]
for _ in range(max(passos_dif)):
    Cs.append(difusao_2d(Cs[-1]))

fig, axes = plt.subplots(1,4,figsize=(16,4))
for ax, p in zip(axes, passos_dif):
    im = ax.imshow(Cs[p], cmap='hot', vmin=0)
    ax.set_title(f'Passo {p}'); ax.axis('off')
plt.suptitle('Difusão 2D por Autômato Celular'); plt.tight_layout(); plt.show()


### 📝 Exercício Final

Modifique a simulação de difusão para ter **2 pontos** de concentração máxima (em posições opostas). Observe a interação entre as duas frentes de difusão.

In [ ]:
# ✏️ Dois pontos de difusão:
C2 = np.zeros((N,N))
C2[N//4, N//4] = 1.0
C2[3*N//4, 3*N//4] = 1.0  # segundo ponto

Cs2 = [C2.copy()]
for _ in range(80):
    Cs2.append(difusao_2d(Cs2[-1]))

fig, axes = plt.subplots(1,4,figsize=(16,4))
for ax,p in zip(axes,[0,10,30,80]):
    ax.imshow(Cs2[p],cmap='hot',vmin=0); ax.set_title(f'Passo {p}'); ax.axis('off')
plt.suptitle('Difusão com 2 Fontes'); plt.tight_layout(); plt.show()
